[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/projects/03_clustering/project.ipynb)

# Project 03 — Phân khúc khách hàng (Clustering + PCA)

> Một trung tâm thương mại muốn chia khách hàng thành vài **phân khúc** để thiết kế chương trình ưu đãi riêng. Không có nhãn sẵn — bạn phải tự tìm cấu trúc, chọn số cụm có lý do, đặt tên phân khúc dễ hiểu và đề xuất hành động. (Dữ liệu mô phỏng có cột `true_segment` — **chỉ dùng ở cuối** để kiểm chứng; thực tế không có cột này.)

**Bạn sẽ làm**:
1. EDA và chuẩn hoá đặc trưng.
2. Chọn số cụm $k$ bằng elbow, silhouette, Davies–Bouldin.
3. K-Means: hồ sơ từng cụm, đặt tên; kiểm tra **độ ổn định** theo khởi tạo.
4. Gaussian Mixture: chọn $k$ bằng BIC; gán mềm — tìm khách "lưng chừng" giữa hai phân khúc.
5. PCA để trực quan hoá; kiểm chứng bằng nhãn thật (ARI); báo cáo hành động marketing.

**Tiêu chí đạt**: chọn đúng $k = 5$ bằng silhouette · silhouette > 0.4 · ARI(K-Means) > 0.85 · BIC chọn $k = 5$ và ARI(GMM) > 0.95.

**Chương liên quan**: 03 (EDA), 04 (chuẩn hoá), 20 (K-Means, chọn k, phân cụm phân cấp), 21 (PCA), 23 (GMM, EM).

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/projects/03_clustering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

customers = pd.read_csv(repo_path("data", "sample", "customer_segments.csv"))
FEATURES = ["annual_income_musd", "spending_score", "age", "visits_per_month"]
print(customers.shape)
customers[FEATURES].describe().round(1)

## Bước 1 — Chuẩn hoá

K-Means và GMM dựa trên khoảng cách → các đặc trưng cần cùng thang đo. Chuẩn hoá `FEATURES` bằng `StandardScaler` → `X` (numpy array). Vẽ ma trận scatter (`pd.plotting.scatter_matrix`) để nhìn cấu trúc.

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
X = StandardScaler().fit_transform(customers[FEATURES])
### END CODE HERE ###
pd.plotting.scatter_matrix(customers[FEATURES], figsize=(8, 8), s=8, alpha=0.5)
plt.show()

In [ ]:
check_equal(X.shape, (600, 4), "kích thước")
check_close(X.mean(axis=0), np.zeros(4), tol=1e-9, message="trung bình 0")
check_close(X.std(axis=0), np.ones(4), tol=1e-9, message="độ lệch chuẩn 1")

## Bước 2 — Chọn số cụm

Với $k = 2, \ldots, 8$: chạy `KMeans(k, n_init=10, random_state=0)`, ghi inertia, silhouette, Davies–Bouldin vào `k_table` (DataFrame index $k$, cột `inertia`, `silhouette`, `davies_bouldin`). Chọn `chosen_k` = $k$ có silhouette lớn nhất.

In [ ]:
### START CODE HERE ### (≈ 7 dòng)
rows = {}
for k in range(2, 9):
    km = KMeans(k, n_init=10, random_state=0).fit(X)
    rows[k] = {"inertia": km.inertia_, "silhouette": silhouette_score(X, km.labels_), "davies_bouldin": davies_bouldin_score(X, km.labels_)}
k_table = pd.DataFrame(rows).T
chosen_k = int(k_table.silhouette.idxmax())
### END CODE HERE ###
print(k_table.round(3), "\nchọn k =", chosen_k)
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, col in zip(axes, k_table.columns):
    ax.plot(k_table.index, k_table[col], marker="o")
    ax.set_title(col)
    ax.set_xlabel("k")
plt.tight_layout()
plt.show()

In [ ]:
check_equal(list(k_table.index), list(range(2, 9)), "k = 2..8")
check_equal(chosen_k, 5, "silhouette chọn k = 5")
check_greater(k_table.loc[5, "silhouette"], 0.4, "silhouette tại k = 5")
check_true(k_table.inertia.is_monotonic_decreasing, "inertia luôn giảm theo k (vì vậy không dùng inertia nhỏ nhất để chọn k)")

## Bước 3 — K-Means, hồ sơ cụm, độ ổn định

Huấn luyện `kmeans = KMeans(chosen_k, n_init=10, random_state=0)` → nhãn `labels`. Lập `profile` = trung bình **đặc trưng gốc** (chưa chuẩn hoá) theo cụm, kèm cột `n` (số khách). Sau đó kiểm tra độ ổn định: với 10 seed, chạy `KMeans(chosen_k, n_init=1, random_state=seed)` và tính ARI với `labels` → mảng `stability`.

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
kmeans = KMeans(chosen_k, n_init=10, random_state=0).fit(X)
labels = kmeans.labels_
profile = customers[FEATURES].assign(cluster=labels).groupby("cluster").mean()
profile["n"] = np.bincount(labels)
stability = np.array([adjusted_rand_score(labels, KMeans(chosen_k, n_init=1, random_state=seed).fit(X).labels_) for seed in range(10)])
### END CODE HERE ###
print(profile.round(1))
print("ARI với nghiệm n_init=10 khi chỉ khởi tạo 1 lần:", np.round(stability, 3))

In [ ]:
check_equal(int(profile.n.sum()), 600, "mọi khách được gán cụm")
check_equal(list(profile.columns), FEATURES + ["n"], "các cột hồ sơ")
check_less(stability.min(), 0.8, "một khởi tạo đơn lẻ có thể kẹt ở cực tiểu địa phương")
check_greater(np.median(stability), 0.9, "đa số khởi tạo cho nghiệm gần giống")

**Đặt tên mẫu** (đọc bảng `profile`): thu nhập thấp – chi tiêu thấp – lớn tuổi → *"tiết kiệm"*; thu nhập trung bình – chi tiêu trung bình → *"trung lưu"*; thu nhập cao – chi tiêu thấp → *"giàu thận trọng"*; thu nhập thấp – chi tiêu cao – trẻ – đến thường xuyên → *"trẻ bốc đồng"*; thu nhập cao – chi tiêu cao – đến rất thường xuyên → *"chi tiêu mạnh"*. Độ ổn định: một lần khởi tạo đôi khi kẹt ở nghiệm kém (ARI ~0.6) — luôn dùng `n_init` ≥ 10 và kiểm tra ổn định trước khi báo cáo phân khúc.

## Bước 4 — Gaussian Mixture

GMM cho phép cụm hình elip và **gán mềm** (xác suất thuộc mỗi cụm). Với $k = 2..8$, fit `GaussianMixture(k, n_init=3, random_state=0)` và ghi BIC → `bic` (Series index $k$); `gmm_k` = $k$ có BIC nhỏ nhất. Fit lại `gmm` với `gmm_k` (`n_init=5`), lấy `gmm_labels` và xác suất `proba`; đếm số khách "lưng chừng" (xác suất cụm lớn nhất < 0.8) → `n_uncertain`.

In [ ]:
### START CODE HERE ### (≈ 6 dòng)
bic = pd.Series({k: GaussianMixture(k, n_init=3, random_state=0).fit(X).bic(X) for k in range(2, 9)})
gmm_k = int(bic.idxmin())
gmm = GaussianMixture(gmm_k, n_init=5, random_state=0).fit(X)
gmm_labels = gmm.predict(X)
proba = gmm.predict_proba(X)
n_uncertain = int((proba.max(axis=1) < 0.8).sum())
### END CODE HERE ###
print(bic.round(0).to_dict(), "→ BIC chọn k =", gmm_k, f"| khách lưng chừng: {n_uncertain}")

In [ ]:
check_equal(gmm_k, 5, "BIC chọn k = 5")
check_equal(proba.shape, (600, 5), "ma trận xác suất")
check_close(proba.sum(axis=1), np.ones(600), message="xác suất mỗi khách cộng lại bằng 1")
check_less(n_uncertain, 60, "phần lớn khách thuộc rõ một phân khúc")

## Bước 5 — Trực quan hoá và kiểm chứng

Chiếu `X` xuống 2 chiều bằng PCA → `Z`, tỉ lệ phương sai giải thích `explained`; vẽ hai biểu đồ: tô màu theo K-Means và theo GMM. Cuối cùng (chỉ vì dữ liệu mô phỏng có nhãn thật) tính `ari_kmeans`, `ari_gmm` so với `true_segment`.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
# STARTER: Z, explained, ari_kmeans, ari_gmm = None, None, None, None
pca = PCA(2).fit(X)
Z = pca.transform(X)
explained = pca.explained_variance_ratio_
ari_kmeans, ari_gmm = adjusted_rand_score(customers.true_segment, labels), adjusted_rand_score(customers.true_segment, gmm_labels)
### END CODE HERE ###
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, lab, title in [(axes[0], labels, f"K-Means (ARI {ari_kmeans:.2f})"), (axes[1], gmm_labels, f"GMM (ARI {ari_gmm:.2f})")]:
    ax.scatter(Z[:, 0], Z[:, 1], c=lab, cmap="tab10", s=10)
    ax.set_title(title)
    ax.set_xlabel(f"PC1 ({explained[0]:.0%})")
    ax.set_ylabel(f"PC2 ({explained[1]:.0%})")
plt.tight_layout()
plt.show()

In [ ]:
check_greater(explained.sum(), 0.8, "2 thành phần giữ > 80% phương sai")
check_greater(ari_kmeans, 0.85, "K-Means gần với phân khúc thật")
check_greater(ari_gmm, 0.95, "GMM gần như khôi phục đúng phân khúc thật")
check_greater(ari_gmm, ari_kmeans, "GMM tốt hơn K-Means trên dữ liệu này")

**Vì sao GMM tốt hơn?** Các phân khúc thật có **độ phân tán khác nhau** theo từng đặc trưng (cụm elip, kích thước khác nhau); K-Means ngầm giả định cụm tròn, cùng cỡ nên cắt sai ở biên giữa "trung lưu" và các cụm lân cận. GMM với ma trận hiệp phương sai đầy đủ mô hình hoá được hình dạng đó, và BIC chọn đúng số cụm.

## Bước 6 — Báo cáo hành động

Viết bảng 5 phân khúc: tên, đặc điểm (số liệu từ `profile`), quy mô, **một** hành động marketing cụ thể cho mỗi phân khúc; cùng 2–3 câu về hạn chế (chỉ 4 đặc trưng, ảnh chụp một thời điểm, cần đo hiệu quả hành động bằng A/B test).

✍️ Báo cáo:

**Báo cáo mẫu:**

| Phân khúc | Đặc điểm | Hành động |
|---|---|---|
| Chi tiêu mạnh (~90 khách) | thu nhập & chi tiêu cao, đến ~8 lần/tháng | chương trình VIP, ưu đãi sớm hàng mới |
| Giàu thận trọng (~100) | thu nhập cao, chi tiêu thấp, lớn tuổi | sản phẩm chất lượng/bền, tư vấn cá nhân |
| Trung lưu (~160) | mọi chỉ số trung bình | tích điểm, combo gia đình |
| Trẻ bốc đồng (~115) | trẻ, thu nhập thấp, chi tiêu cao | flash sale, trả góp, mạng xã hội |
| Tiết kiệm (~135) | thu nhập & chi tiêu thấp | khuyến mãi giá, hàng thiết yếu |

Phân khúc ổn định (silhouette 0.41, BIC và silhouette cùng chọn 5 cụm, ổn định theo khởi tạo). Hạn chế: chỉ 4 đặc trưng, một thời điểm; rất ít khách "lưng chừng" (xác suất < 0.8) — có thể nhận thông điệp trung tính. Hiệu quả từng hành động cần đo bằng thí nghiệm, không suy từ phân cụm.